# 🎓 Fine-Tuning SARG LLM on Socratic Tutoring Dialogues

This notebook contains the complete pipeline to fine-tune a pre-trained **Qwen-2.5-7B-Instruct** or **Llama-3.1-8B-Instruct** model on the **SARG LLM Socratic tutoring dialogue dataset** using **Unsloth** and Hugging Face **TRL** (SFTTrainer).

### Prerequisite: Enable GPU
Go to **Runtime > Change runtime type > T4 GPU** (or L4 / A100 if you have Colab Pro).

## 1. Install Unsloth and Dependencies
First, we install `unsloth` and the necessary fine-tuning libraries. This is optimized to run extremely fast on Colab.

> [!IMPORTANT]
> **CRITICAL STEP**: After running this installation cell, you **MUST** go to **Runtime > Restart Session** in the top menu. This resets Python's memory namespace so that the newly installed libraries don't trigger pickling errors during training.

In [ ]:
# Install Unsloth and compatible dependencies
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets transformers

## 2. Load the Pre-trained Model & Tokenizer
We load the base model in 4-bit precision to save GPU memory.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Supports any rope scaling automatically
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 3. Configure LoRA Adapters
We configure LoRA adapters targeting all attention and linear modules. This allows the model to adapt specifically to SARG LLM's Socratic dialogue structure.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank (suggested: 8, 16, 32, 64)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized by Unsloth
    bias = "none",    # Optimized by Unsloth
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 4. Load & Format the Dataset
We load our formatted JSONL dataset `sarg_llama_synthetic_compiled.jsonl` (or `sarg_llama_synthetic_formatted.jsonl`). Since it is structured with the standard OpenAI messages list, we can apply the chat template directly to format the conversations into training tokens.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Set the Qwen-2.5 instruction chat template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convs]
    return { "text" : texts }

# Make sure to upload 'sarg_llama_synthetic_compiled.jsonl' to Colab first!
dataset = load_dataset("json", data_files="sarg_llama_synthetic_compiled.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

## 5. Initialize SFTTrainer and Train
We use Hugging Face TRL `SFTTrainer` and `SFTConfig` to train the model. Unsloth's implementation speeds up training by 2x-5x with lower VRAM consumption.

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training faster for short sequences
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 120, # Set to e.g. 500 or 1000 for a full training run
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 6. Save Model to GGUF (Quantized 4-bit)
Once training is complete, we save the fine-tuned LoRA weights and compile it into a quantized GGUF format, which can be directly deployed locally via **Ollama**, **Llama.cpp**, or **vLLM**.

In [ ]:
# Save LoRA adapters locally
model.save_pretrained("socratic_tutor_lora")
tokenizer.save_pretrained("socratic_tutor_lora")

# Export to 4-bit quantized GGUF format for Ollama
model.save_pretrained_gguf("socratic_tutor_gguf", tokenizer, quantization_method = "q4_k_m")
print("Success! Merged GGUF model saved in socratic_tutor_gguf/")